# Acceptance: calibration targets on transformed outputs

**This is a sign-off page, not a tutorial.** It checks that a calibration target
can name an output of an `OutputSet` — here a 7-day rolling sum of new
infections — and be scored correctly (plan `plans/bayesian-output-targets.plan.md`).

Before this change, `BayesianModel(..., outputs=outputs)` built its save plan from
the targets first. A target naming an output raised `KeyError`. Working around it
was worse: the target pinned the output's saved input to the observation dates,
so a rolling sum was taken over seven *observations* instead of seven *days*,
silently. Now the outputs are saved on the run's own grid, and a target that names
one is read from the evaluated output at its dates.

The model is the summer4template example: an age-stratified SIR with homogeneous
mixing, as in summer3's template, whose target is the weekly rolling sum of
infections. Nothing here asserts; the checks live in
`tests/test_bayesian_output_targets.py`. Run every cell top to bottom (about
20 seconds, mostly sampling). Each section ends with a **What to check** box.

| | You are checking that… |
|---|---|
| A | the target is a weekly rolling sum, and the saved plan keeps the full daily grid |
| B | the old observation-time save would have given a different, wrong weekly curve |
| C | sampling against the weekly target recovers the true parameters, with a progress bar |

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Split,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import BayesianModel, ForceOfInfection, Poisson, Scenario, Uniform

state = Property("disease_state", ("S", "I", "R"))
age = Property("age", ("child", "adult"))
model = FlowModel(PropertyMap.from_property(state).stratify(age))
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("contact_rate")),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / Param("recovery_time")))
model.set_initial_population(
    {age["child"]: 1000.0, age["adult"]: 1500.0},
    [Split(state, {"S": 0.9, "I": 0.1, "R": 0.0})],
)
cm = model.compile()

TRUTH = {"contact_rate": 0.2, "recovery_time": 20.0}
run_kwargs = dict(t0=0.0, t1=183.0, dt=1.0)  # a daily grid, so rolling(7) is seven days
obs_days = np.arange(7.0, 63.0, 7.0)

outputs = OutputSet()
outputs["weekly_infections"] = FlowMass("infection").sum_by().rolling(7, how="sum")

## A. The target and the model's weekly output

Synthetic data: the weekly rolling sum from a run at the true parameters, read
every seventh day, with 1% multiplicative noise (as in summer3's template). The
target names the output, with no quantity of its own. The plot shows the weekly
output over the whole run and the observations on it; the table shows the save
plan `BayesianModel` built.

In [ ]:
truth = outputs.evaluate(cm.run(TRUTH, save=outputs.plan(), **run_kwargs), TRUTH)
weekly = truth["weekly_infections"]
true_at_obs = np.asarray(weekly.at_times(obs_days).values).reshape(-1)
observed = true_at_obs * np.exp(np.random.default_rng(0).normal(scale=0.01, size=obs_days.size))

target = Target(
    key="weekly_infections", times=obs_days, values=observed, likelihood=Poisson("sum")
)
bm = BayesianModel(
    cm,
    {},
    [Uniform("contact_rate", 0.001, 0.5), Uniform("recovery_time", 10.0, 50.0)],
    TargetSet((target,)),
    outputs=outputs,
    run_kwargs=run_kwargs,
)

days = np.asarray(weekly.times.values)
fig = pd.Series(
    np.asarray(weekly.values).reshape(-1), index=pd.Index(days, name="day"), name="model"
).plot(
    title="A. Weekly infections (7-day rolling sum) and the observations",
    labels={"value": "infections in the past 7 days", "variable": ""},
)
fig.add_scatter(x=obs_days, y=observed, mode="markers", marker={"size": 9}, name="observed")
fig.show()
pd.DataFrame(
    [
        {"save key": key, "saves": repr(req.what), "save times": "run grid" if req.ts is None else req.ts}
        for key, req in bm.save_plan.requests.items()
    ]
)

> **What to check (A).** The observations sit on the model's weekly curve
> (within the 1% noise). The save plan has one request, the infection flow,
> saved on the run grid — not at the eight observation days.

## B. What pinning the save to the observation days computes

This reproduces what the old plan did once the `KeyError` was worked around: the
infection flow saved only on the eight observation days, then the same
`rolling(7)` output evaluated from it. The plot compares that with the correct
weekly output at the same days.

In [ ]:
pinned = SavePlan(
    requests={"weekly_infections": SaveRequest(FlowMass("infection"), ts=obs_days)}
)
wrong = outputs.evaluate(cm.run(TRUTH, save=pinned, **run_kwargs), TRUTH)["weekly_infections"]
wrong_at_obs = np.asarray(wrong.at_times(obs_days).values).reshape(-1)

fig = pd.DataFrame(
    {"correct (daily grid)": true_at_obs, "pinned to observation days": wrong_at_obs},
    index=pd.Index(obs_days, name="day"),
).plot(
    markers=True,
    title="B. Weekly infections: the daily grid vs a save pinned to observation days",
    labels={"value": "infections in the past 7 days", "variable": ""},
)
fig.show()
pd.DataFrame(
    {"correct": true_at_obs, "pinned": wrong_at_obs}, index=pd.Index(obs_days, name="day")
).round(1)

> **What to check (B).** The pinned curve is far from the correct one: it is
> blank (NaN) until seven observations exist, then sums seven *weeks* of daily
> rates. The correct column matches the curve in A.

## C. Sampling against the weekly target

`bm.sample` with `progress_bar=True` prints a bar for warmup and sampling (one bar
for all four chains, which run vectorized; the first few seconds are compilation,
before the bar appears). The scatter shows the posterior draws with the true
parameters; the second plot is the posterior band of the weekly output against
the observations.

In [ ]:
run = bm.sample(num_warmup=300, num_samples=300, num_chains=4, seed=0, progress_bar=True)

In [ ]:
draws = pd.DataFrame({k: np.asarray(run.samples[k]).reshape(-1) for k in TRUTH})
fig = draws.plot.scatter(
    x="contact_rate",
    y="recovery_time",
    opacity=0.4,
    title="C. Posterior draws and the true parameters",
)
fig.add_scatter(
    x=[TRUTH["contact_rate"]],
    y=[TRUTH["recovery_time"]],
    mode="markers",
    marker={"symbol": "star", "size": 18, "color": "gold", "line": {"width": 1}},
    name="truth",
)
fig.update_layout(xaxis_title="contact_rate", yaxis_title="recovery_time (days)")
fig.show()

band = bm.posterior_runs(run, n=100, seed=1, scenarios={"posterior": Scenario()})
quantiles = band.quantiles(q=(0.025, 0.5, 0.975))["posterior"]
ribbon = pd.DataFrame(
    {
        "2.5%": quantiles[("weekly_infections", "0.025")].to_numpy(),
        "median": quantiles[("weekly_infections", "0.5")].to_numpy(),
        "97.5%": quantiles[("weekly_infections", "0.975")].to_numpy(),
    },
    index=pd.Index(np.asarray(quantiles.index), name="day"),
)
fig = ribbon.plot(
    title="C. Posterior 95% band of weekly infections, with the observations",
    labels={"value": "infections in the past 7 days", "variable": ""},
)
fig.add_scatter(x=obs_days, y=observed, mode="markers", marker={"size": 9, "color": "black"}, name="observed")
fig.show()

summary = draws.quantile([0.025, 0.5, 0.975]).T
summary.columns = ["2.5%", "median", "97.5%"]
summary["truth"] = pd.Series(TRUTH)
summary.round(3)

> **What to check (C).** A progress bar appeared under the sampling cell and
> reached 100%. The gold star sits inside the cloud of draws, and in the table
> each true value lies between the 2.5% and 97.5% columns. The observations lie
> inside the posterior band.